# 22 (DS) — Supervised ML

**Data Scientist perspective.** The full ml_scope §73 success-criteria workflow: feature engineering → train a Spark-compatible estimator (numpy fit, SQL-pushdown inference) → evaluate in SQL → and an IntegratedML AutoML alternative. All on the Param-backed ML framework.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Training data

A small credit-style dataset: income and age predict default.

In [ ]:
import pandas as pd

pdf = pd.DataFrame({
    "renda": [3000, 5000, 7000, 9000, 11000, 13000, 15000, 17000, 19000, 21000],
    "idade": [25, 30, 35, 40, 45, 50, 55, 60, 65, 70],
    "inadimplente": [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
})
df = session.createDataFrame(pdf)
df.show()

## 2. Feature engineering

Standardize the numeric features, then assemble.

In [ ]:
from irispark.ml.feature import StandardScaler, VectorAssembler

scaled = StandardScaler(inputCol="renda", outputCol="renda_std", withMean=True, withStd=True).fit(df).transform(df)
scaled = StandardScaler(inputCol="idade", outputCol="idade_std", withMean=True, withStd=True).fit(scaled).transform(scaled)
feats = VectorAssembler(inputCols=["renda_std", "idade_std"], outputCol="features").transform(scaled)
feats.select("renda_std", "idade_std", "inadimplente").show()

## 3. Logistic regression (numpy fit, SQL predict)

`fit()` trains in numpy; `transform()` pushes coefficients back as a SQL sigmoid — scoring never moves data to Python.

In [ ]:
from irispark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol=["renda_std", "idade_std"], labelCol="inadimplente", maxIter=500, learningRate=0.5)
model = lr.fit(feats)
pred = model.transform(feats)
pred.select("renda_std", "inadimplente", "probability", "prediction").show()

## 4. Evaluate in SQL

Accuracy and ROC AUC via the SQL evaluators.

In [ ]:
from irispark.ml.evaluation import BinaryClassificationEvaluator

acc = BinaryClassificationEvaluator(predictionCol="prediction", labelCol="inadimplente", metricName="accuracy").evaluate(pred)
auc = BinaryClassificationEvaluator(predictionCol="probability", labelCol="inadimplente", metricName="areaUnderROC").evaluate(pred)
print(f"accuracy={acc:.3f}  auc={auc:.3f}")

## 5. Linear regression (numpy fit, SQL predict)

Predict a continuous target with the same pattern.

In [ ]:
from irispark.ml.regression import LinearRegression
from irispark.ml.evaluation import RegressionEvaluator

pdf2 = pd.DataFrame({"x": [1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0], "y": [3.0,5.0,7.0,9.0,11.0,13.0,15.0,17.0]})
df2 = session.createDataFrame(pdf2)
lin = LinearRegression(featuresCol=["x"], labelCol="y").fit(df2)
lin_pred = lin.transform(df2)
rmse = RegressionEvaluator(predictionCol="prediction", labelCol="y", metricName="rmse").evaluate(lin_pred)
print(f"linear rmse={rmse:.6f}  (should be ~0)")
lin_pred.show()

## 6. IntegratedML AutoML (IRIS extension)

AutoML trains a model automatically via IntegratedML. It needs enough training rows (the provider returns `NoEstimatorChosen` on tiny samples), so we generate a 200-row dataset here. Training runs server-side; the client polls for completion.

In [ ]:
import numpy as np
from irispark.ml.automl import AutoMLClassifier

rng = np.random.default_rng(0)
X = rng.normal(size=(200, 2))
y = (X[:, 0] + X[:, 1] > 0).astype(float)
am_df = session.createDataFrame(
    [(float(x0), float(x1), float(yi)) for x0, x1, yi in zip(X[:, 0], X[:, 1], y)],
    ["x1", "x2", "label"],
)

am = AutoMLClassifier(featuresCol=["x1", "x2"], labelCol="label", maxTime=60).fit(am_df)
print("AutoML model:", am.modelName)
am_pred = am.transform(am_df)
am_pred.select("x1", "x2", "label", "prediction").show()
session.sql("DROP MODEL " + am.modelName)

## 7. Hyperparameter tuning

`CrossValidator` picks the best `regParam` over a grid using 3-fold cross-validation, then retrains on the full dataset.

In [ ]:
from irispark.ml.tuning import ParamGridBuilder, CrossValidator
from irispark.ml.evaluation import BinaryClassificationEvaluator

lr_base = LogisticRegression(featuresCol=["x1", "x2"], labelCol="label", maxIter=300, learningRate=0.5)
grid = ParamGridBuilder()
grid.addGrid(lr_base.getParam("regParam"), [0.0, 0.1, 1.0])
eval_ = BinaryClassificationEvaluator(predictionCol="prediction", labelCol="label", metricName="accuracy")
cv = CrossValidator(estimator=lr_base, estimatorParamMaps=grid.build(), evaluator=eval_, numFolds=3)
best = cv.fit(am_df)
print("best params:", cv.bestParams, "| avg metrics:", cv.avgMetrics)
best.transform(am_df).select("x1", "label", "prediction").show(5)

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")